In [3]:

import pandas as pd
from pathlib import Path

# Path to the 2011 Census file
file_path = Path("../data/raw/census_2011/DDW27C-01_MDDS.xls")

# Verify that the file exists
if not file_path.exists():
    raise FileNotFoundError(f"File not found: {file_path.resolve()}")

# Display worksheet names
excel_file = pd.ExcelFile(file_path)
print("Worksheets:", excel_file.sheet_names)

# Read the first 25 rows without assuming column headers
sheet_name = excel_file.sheet_names[0]

df_preview = pd.read_excel(
    file_path,
    sheet_name=sheet_name,
    header=None,
    nrows=25
)

print("\nPreview shape:", df_preview.shape)
print("\nFirst 25 rows:")
print(df_preview.to_string(index=False, header=False))



Worksheets: ['C01']

Preview shape: (25, 34)

First 25 rows:
  NaN   NaN    NaN    NaN    NaN                      NaN    NaN C -1 POPULATION BY RELIGIOUS COMMUNITY - 2011      NaN      NaN                    NaN      NaN      NaN      NaN     NaN     NaN       NaN    NaN     NaN C -1 POPULATION BY RELIGIOUS COMMUNITY    NaN     NaN      NaN     NaN     NaN     NaN    NaN     NaN                                                       NaN   NaN     NaN                 NaN    NaN     NaN
Table State Distt. Tehsil   Town                Area Name Total/                                           NaN      NaN      NaN  Religious communities      NaN      NaN      NaN     NaN     NaN       NaN    NaN     NaN                                    NaN    NaN     NaN      NaN     NaN     NaN     NaN    NaN     NaN                                                       NaN   NaN     NaN                 NaN    NaN     NaN
 Name  Code   Code   Code   Code                      NaN Rural/                 

In [4]:

import pandas as pd
from pathlib import Path

# 1. File paths
input_path = Path("../data/raw/census_2011/DDW27C-01_MDDS.xls")
output_path = Path("../data/processed/census_2011_cleaned.csv")

# Create processed folder if it doesn't exist
output_path.parent.mkdir(parents=True, exist_ok=True)

# 2. Read the Excel file
df = pd.read_excel(
    input_path,
    sheet_name="C01",
    header=None
)

print("Original shape:", df.shape)

# 3. Remove report headings
# Based on your preview, row 7 (index 7) contains Maharashtra Total
df = df.iloc[7:].reset_index(drop=True)

# 4. Assign column names
columns = [
    "Table_Code", "State_Code", "District_Code", "Tehsil_Code",
    "Town_Code", "Area_Name", "Area_Type",
    "Total_Persons", "Total_Males", "Total_Females",
    "Hindu_Persons", "Hindu_Males", "Hindu_Females",
    "Muslim_Persons", "Muslim_Males", "Muslim_Females",
    "Christian_Persons", "Christian_Males", "Christian_Females",
    "Sikh_Persons", "Sikh_Males", "Sikh_Females",
    "Buddhist_Persons", "Buddhist_Males", "Buddhist_Females",
    "Jain_Persons", "Jain_Males", "Jain_Females",
    "Other_Religions_Persons", "Other_Religions_Males",
    "Other_Religions_Females",
    "Religion_Not_Stated_Persons",
    "Religion_Not_Stated_Males",
    "Religion_Not_Stated_Females"
]

df.columns = columns

# 5. Remove completely empty rows
df = df.dropna(how="all").reset_index(drop=True)

# 6. Clean text columns
text_columns = [
    "Table_Code", "Area_Name", "Area_Type"
]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

# 7. Convert population and code columns to numeric
numeric_columns = [col for col in columns if col not in text_columns]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 8. Save cleaned data
df.to_csv(output_path, index=False)

print("Cleaned shape:", df.shape)
print("Saved to:", output_path.resolve())

# 9. Preview
print(df[["Area_Name", "Area_Type", "Total_Persons"]].head(10).to_string(index=False))


Original shape: (1722, 34)
Cleaned shape: (1715, 34)
Saved to: C:\Users\Aditya\Desktop\Maharashtra_Religion_Dashboard\data\processed\census_2011_cleaned.csv
               Area_Name Area_Type  Total_Persons
     State - MAHARASHTRA     Total      112374333
     State - MAHARASHTRA     Rural       61556074
     State - MAHARASHTRA     Urban       50818259
Greater Mumbai (M Corp.)     Urban       12442373
    District - Nandurbar     Total        1648295
    District - Nandurbar     Rural        1372821
    District - Nandurbar     Urban         275474
Sub-District - Akkalkuwa     Total         245861
Sub-District - Akkalkuwa     Rural         215974
Sub-District - Akkalkuwa     Urban          29887


In [5]:

import pandas as pd
from pathlib import Path

file_path = Path("../data/processed/census_2011_cleaned.csv")
df = pd.read_csv(file_path)

maharashtra = df[
    df["Area_Name"].str.strip().str.upper()
    == "STATE - MAHARASHTRA"
]

print(maharashtra[["Area_Name", "Area_Type", "Total_Persons"]].to_string(index=False))

total = maharashtra.loc[
    maharashtra["Area_Type"] == "Total", "Total_Persons"
].sum()

rural = maharashtra.loc[
    maharashtra["Area_Type"] == "Rural", "Total_Persons"
].sum()

urban = maharashtra.loc[
    maharashtra["Area_Type"] == "Urban", "Total_Persons"
].sum()

print("\nState total:", total)
print("Rural:", rural)
print("Urban:", urban)
print("Rural + Urban:", rural + urban)
print("Difference:", total - rural - urban)


          Area_Name Area_Type  Total_Persons
State - MAHARASHTRA     Total      112374333
State - MAHARASHTRA     Rural       61556074
State - MAHARASHTRA     Urban       50818259

State total: 112374333
Rural: 61556074
Urban: 50818259
Rural + Urban: 112374333
Difference: 0


In [6]:

import pandas as pd
from pathlib import Path

files = {
    "Census 2001": Path("../data/processed/census_2001_cleaned.csv"),
    "Census 2011": Path("../data/processed/census_2011_cleaned.csv")
}

for year, file_path in files.items():
    if file_path.exists():
        df = pd.read_csv(file_path)
        print(f"\n{year}")
        print("Shape:", df.shape)
        print("Columns:", len(df.columns))
        print("File exists: YES")
    else:
        print(f"\n{year}: FILE NOT FOUND")



Census 2001
Shape: (1547, 34)
Columns: 34
File exists: YES

Census 2011
Shape: (1715, 34)
Columns: 34
File exists: YES
